# TBD Phase 2: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: Single-node processing speed, parallel execution, and memory usage.
- **Scalability**: How performance changes with the number of cores (single-node) and executors (cluster).
- **Computing Models**: Out-of-core vs. In-memory processing, and Eager vs. Lazy execution.

### Engine Capabilities
The following table summarizes the key capabilities of the engines we will be testing. Use this as a reference.

| Engine | Query Optimizer | Distributed | Arrow-backed | Out-of-Core | Parallel | APIs | GPU Support |
|---|---|---|---|---|--|---|---|
| **Pandas** | ❌ | ❌ | optional ≥ 2.0 | ❌ | ❌ | DataFrame | ❌ |
| **Polars** | ✅ | ❌ | ✅ | ✅ | ✅ | DataFrame | ✅ (opt) |
| **PySpark** | ✅ | ✅ | Pandas UDF/IO | ✅ | ✅ | SQL, DataFrame | ❌ (no GPU) |
| **DuckDB** | ✅ | ❌ | ✅ | ✅ | ❌ | SQL, Relational API | ❌ |

## Prerequisites
Ensure you have the necessary libraries installed.

In [1]:
# %pip install polars pandas duckdb pyspark faker deltalake memory_profiler pyarrow

In [1]:
import polars as pl
import pandas as pd
import duckdb
from pyspark.sql import SparkSession
from faker import Faker
import numpy as np
import os
import time
import psutil
from memory_profiler import memory_usage
import matplotlib.pyplot as plt
import copy

In [ ]:
# Initialize Spark (Single Node)
spark = SparkSession.builder \
    .appName("BigDataLab2") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

## Part 1: Data Generation

We will generate a synthetic dataset simulating social media posts with a rich schema.

**Schema**:
- `post_id` (String): Unique identifier.
- `user_id` (Integer): User identifier.
- `timestamp` (DateTime): Time of post.
- `content` (String): Text content.
- `likes` (Integer): Number of likes.
- `views` (Integer): Number of views.
- `category` (String): Post category.
- `tags` (List[String]): Hashtags.
- `location` (String): User location.
- `device` (String): Device used (Mobile, Web, etc.).
- `latency` (Float): Network latency.
- `error_rate` (Float): Error rate during upload.

In [2]:
def generate_data(num_records=1_000_000, output_path="/media/mikic202/Nowy1/uczelnia/semestr_9/social_media_data.parquet"):
    fake = Faker()

    print(f"Generating {num_records} records...")

    # Generate data using numpy for speed where possible
    data = {
        "post_id": [fake.uuid4() for _ in range(num_records)],
        "user_id": np.random.randint(1, 100_000, num_records),
        "timestamp": pd.date_range(start="2023-01-01", periods=num_records, freq="s").to_numpy().astype("datetime64[us]"),
        "likes": np.random.randint(0, 10_000, num_records),
        "views": np.random.randint(0, 1_000_000, num_records),
        "category": np.random.choice(["Tech", "Health", "Travel", "Food", "Fashion", "Politics", "Sports"], num_records),
        "tags": [np.random.choice(["#viral", "#new", "#trending", "#hot", "#update"], size=np.random.randint(1, 4)).tolist() for _ in range(num_records)],
        "location": np.random.choice(["USA", "UK", "DE", "PL", "FR", "JP", "BR"], num_records),
        "device": np.random.choice(["Mobile", "Desktop", "Tablet"], num_records),
        "latency": np.random.uniform(10.0, 500.0, num_records),
        "error_rate": np.random.beta(1, 10, num_records),
        "content": [fake.sentence() for _ in range(min(num_records, 1000))] * (num_records // 1000 + 1)
    }

    # Trim to exact size
    data["content"] = data["content"][:num_records]

    df = pd.DataFrame(data)

    print("Writing to Parquet...")
    df.to_parquet(output_path, engine="pyarrow")
    print(f"Data saved to {output_path}")

# Generate 5 million records
generate_data(num_records=5_000_000)

Generating 5000000 records...
Writing to Parquet...
Data saved to /media/mikic202/Nowy1/uczelnia/semestr_9/social_media_data.parquet


## Part 2: Measuring Performance

### 2.1 Execution Time
Use `%time` or `%timeit` to measure execution time.

In [ ]:
# Example: Measuring time for all engines
print("--- Performance Benchmark Example ---")

# Pandas
print("Pandas Load Time:")
%time df_pd = pd.read_parquet("social_media_data.parquet")

# Polars
print("\nPolars Load Time:")
%time df_pl = pl.read_parquet("social_media_data.parquet")

# DuckDB
print("\nDuckDB Query Time:")
%time duckdb.sql("SELECT count(*) FROM 'social_media_data.parquet'").show()

# PySpark
print("\nSpark Load Time:")
%time df_spark = spark.read.parquet("social_media_data.parquet"); df_spark.count()

## Part 3: Student Tasks

### Task 1: Performance & Scalability (Single Node)

**Goal**: Benchmark the engines and test how they scale with available CPU cores.

**Instructions**:
1.  **Define Queries**: Create 3 distinct queries of your own choice. They should cover:
    -   **Query A**: A simple aggregation (e.g., grouping by a categorical column and calculating means).
    -   **Query B**: A window function or more complex transformation.
    -   **Query C**: A join (e.g., self-join or join with a smaller generated table) with filtering.
2.  **Benchmark**: Implement these queries in **Pandas, Polars, DuckDB, and PySpark**.
    -   Measure **Execution Time** using `%time` or `time.time()`.
    -   Measure **Peak Memory** usage using `memory_profiler` (e.g., `memory_usage()`).
3.  **Scalability Test**: 
    -   SELECT **all engines** that support parallel execution on a single node (e.g., Polars, DuckDB).
    -   Run **all 3 queries** with different numbers of threads/cores (e.g., 1, 2, 4, 8).
    -   Plot the speedup for each query and engine.

**Tip**: 
-   Polars: [polars.thread_pool_size](https://docs.pola.rs/api/python/stable/reference/api/polars.thread_pool_size.html) Please also note that *Thread configuration in Polars requires process restart*
-   DuckDB: `PRAGMA threads=n`
-   Spark: `master="local[n]"`

#### T1.1. Queries definition



---

1. Simple aggregation
```sql
-- SELECT device types and its technical information averages
SELECT
    device,
    count(*) as device_count,
    mean(latency) as avg_altency,
    mean(error_rate) as avg_error_rate
FROM
    db_table
GROUP BY
    device
```

---

2. Window functions
```sql
SELECT
    post_id,
    category,
    DATE(timestamp) AS post_date,
    likes,
    views,
    RANK() OVER (
        PARTITION BY category, DATE(timestamp)
        ORDER BY likes DESC
    ) AS daily_like_rank,
    AVG(views) OVER (
        PARTITION BY category
        ORDER BY timestamp
        ROWS BETWEEN 100 PRECEDING AND CURRENT ROW
    ) AS rolling_avg_views
FROM
    db_table;
```

---

3. JOINs
```sql
-- SELECT all post_ids and get previous post_id of the 1st post uploader (user_id)
WITH posts_ranked AS (
    SELECT
        post_id
        , user_id
        , ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY timestamp DESC
        ) AS userpost_history
    FROM
        db_table
)
SELECT
    cur.user_id as user_id
    , cur.post_id as post_id
    , prev.post_id as prevoius_post_id
FROM
    posts_ranked cur
LEFT JOIN
    posts_ranked prev
ON
    cur.user_id = prev.user_id
    AND cur.userpost_history = prev.userpost_history - 1
```


<!-- ```sql
SELECT
    p1.post_id AS current_post,
    p2.post_id AS previous_post,
    p1.user_id,
    p1.likes AS current_likes,
    p2.likes AS previous_likes,
    p1.timestamp
FROM
    db_table p1
JOIN db_table p2
    ON p1.user_id = p2.user_id
    AND p2.timestamp = (
        SELECT MAX(timestamp)
        FROM db_table
        WHERE user_id = p1.user_id
          AND timestamp < p1.timestamp
   )
WHERE
    p1.likes > p2.likes;
``` -->

---

#### T1.2. Benchmarks

In [1]:
# Your Code Here for Task 1
import utils

TABLE_LOCATION = "/media/mikic202/Nowy1/uczelnia/semestr_9/social_media_data.parquet"
# TABLE_LOCATION = "./social_media_data.parquet"

Pandas

In [ ]:
print(" --- Query 1: --- ")
pandas_tim_q1 = utils.benchmark_pandas_time(df_pd, utils.query_aggregation_pandas)
pandas_mem_q1 = utils.benchmark_pandas_memory(df_pd, utils.query_aggregation_pandas)

print("\n\n --- Query 2 --- ")
pandas_tim_q2 = utils.benchmark_pandas_time(df_pd, utils.query_window_pandas)
pandas_mem_q2 = utils.benchmark_pandas_memory(df_pd, utils.query_window_pandas)

print("\n\n --- Query 3 --- ")
pandas_tim_q3 = utils.benchmark_pandas_time(df_pd, utils.query_join_pandas)
pandas_mem_q3 = utils.benchmark_pandas_memory(df_pd, utils.query_join_pandas)

Polars

In [ ]:
print(" --- Query 1: --- ")
polars_tim_q1 = utils.benchmark_polars_time(df_pl, utils.query_aggregation_polars)
polars_mem_q1 = utils.benchmark_polars_memory(df_pl, utils.query_aggregation_polars)

print("\n\n --- Query 2 --- ")
polars_tim_q2 = utils.benchmark_polars_time(df_pl, utils.query_window_polars)
polars_mem_q2 = utils.benchmark_polars_memory(df_pl, utils.query_window_polars)

print("\n\n --- Query 3 --- ")
polars_tim_q3 = utils.benchmark_polars_time(df_pl, utils.query_join_polars)
polars_mem_q3 = utils.benchmark_polars_memory(df_pl, utils.query_join_polars)

DuckDB

In [ ]:
print(" --- Query 1: --- ")
duck_db_tim_q1 = utils.benchmark_duckdb_time(TABLE_LOCATION, utils.AGGREGATION_QUERY)
duck_db_mem_q1 = utils.benchmark_duckdb_memory(TABLE_LOCATION, utils.AGGREGATION_QUERY)

print("\n\n --- Query 2 --- ")
duck_db_tim_q2 = utils.benchmark_duckdb_time(TABLE_LOCATION, utils.WINDOWFUNCTION_QUERY)
duck_db_mem_q2 = utils.benchmark_duckdb_memory(TABLE_LOCATION, utils.WINDOWFUNCTION_QUERY)

print("\n\n --- Query 3 --- ")
duck_db_tim_q3 = utils.benchmark_duckdb_time(TABLE_LOCATION, utils.JOIN_QUERY)
duck_db_mem_q3 = utils.benchmark_duckdb_memory(TABLE_LOCATION, utils.JOIN_QUERY)

Spark

In [6]:
spark_benchmark_obj = utils.SparkBenchmarkObject(session=spark, df=df_spark)

In [ ]:
print(" --- Query 1: --- ")
spark_tim_q1 = utils.benchmark_spark_time(spark_benchmark_obj=spark_benchmark_obj, query=utils.AGGREGATION_QUERY)
spark_mem_q1 = utils.benchmark_spark_memory(spark_benchmark_obj=spark_benchmark_obj, query=utils.AGGREGATION_QUERY)
print("\n\n --- Query 2 --- ")
spark_tim_q2 = utils.benchmark_spark_time(spark_benchmark_obj=spark_benchmark_obj, query=utils.WINDOWFUNCTION_QUERY)
spark_mem_q2 = utils.benchmark_spark_memory(spark_benchmark_obj=spark_benchmark_obj, query=utils.WINDOWFUNCTION_QUERY)

print("\n\n --- Query 3 --- ")
spark_tim_q3 = utils.benchmark_spark_time(spark_benchmark_obj=spark_benchmark_obj, query=utils.JOIN_QUERY)
spark_mem_q3 = utils.benchmark_spark_memory(spark_benchmark_obj=spark_benchmark_obj, query=utils.JOIN_QUERY)

spark.stop()


#### T1.3. Scalability

In [8]:
MAX_NUMBER_OF_THREADS = 10

**Polars**


Due to Polars requiring a process restart to change the thread configuration, other tests must be run after a kernel restart. The results are saved to a file.

In [ ]:
import utils
import csv
import os

# threads_list = [1, 2, 4, 8, 16]
n_threads = 16
os.environ["POLARS_MAX_THREADS"] = str(n_threads)  # ustawienie liczby wątków

import polars as pl

df_pl = pl.read_parquet("social_media_data.parquet")
filename = "save_scalability_result.csv"


def save_scalability_result(filename: str, cores: int, query_name: str, time_seconds: float):
    """
    Zapisuje wynik benchmarku do pliku CSV, dopisując nowy wiersz.

    Args:
        filename (str): nazwa pliku CSV
        cores (int): liczba wątków użytych w benchmarku
        query_name (str): nazwa testowanej funkcji / zapytania
        time_seconds (float): czas wykonania w sekundach
    """
    file_exists = os.path.isfile(filename)

    with open(filename, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=["cores", "query", "time"])

        if not file_exists:
            writer.writeheader()

        writer.writerow({
            "cores": cores,
            "query": query_name,
            "time": time_seconds
        })

DuckDB

In [ ]:

times_per_core_agregation = utils.scalability_duckdb(TABLE_LOCATION, utils.AGGREGATION_QUERY, max_number_of_threads=MAX_NUMBER_OF_THREADS)
print("Times per core for DuckDB for agregation query:")
for cores, t in times_per_core_agregation.items():
    print(f"Cores: {cores:2}, Time: {t:.4f} seconds")

times_per_core_window = utils.scalability_duckdb(TABLE_LOCATION, utils.WINDOWFUNCTION_QUERY, max_number_of_threads=MAX_NUMBER_OF_THREADS)
print("Times per core for DuckDB for window function query:")
for cores, t in times_per_core_window.items():
    print(f"Cores: {cores:2}, Time: {t:.4f} seconds")

times_per_core_join= utils.scalability_duckdb(TABLE_LOCATION, utils.JOIN_QUERY, max_number_of_threads=MAX_NUMBER_OF_THREADS)
print("Times per core for DuckDB for join query:")
for cores, t in times_per_core_join.items():
    print(f"Cores: {cores:2}, Time: {t:.4f} seconds")


In [ ]:
utils.plot_scalability(
    times_per_core_query_dict=times_per_core_agregation,
    title='DuckDB Aggregation Query Scalability'
)
utils.plot_scalability(
    times_per_core_query_dict=times_per_core_window,
    title='DuckDB Window Query Scalability'
)
utils.plot_scalability(
    times_per_core_query_dict=times_per_core_join,
    title='DuckDB Join Query Scalability'
)


Spark

In [ ]:

times_per_core_agregation = utils.scalability_spark(TABLE_LOCATION, utils.AGGREGATION_QUERY, max_number_of_threads=MAX_NUMBER_OF_THREADS)
print("Times per core for Spark for agregation query:")
for cores, t in times_per_core_agregation.items():
    print(f"Cores: {cores:2}, Time: {t:.4f} seconds")

times_per_core_window = utils.scalability_spark(TABLE_LOCATION, utils.WINDOWFUNCTION_QUERY, max_number_of_threads=MAX_NUMBER_OF_THREADS)
print("Times per core for Spark for window function query:")
for cores, t in times_per_core_window.items():
    print(f"Cores: {cores:2}, Time: {t:.4f} seconds")

times_per_core_join= utils.scalability_spark(TABLE_LOCATION, utils.JOIN_QUERY, max_number_of_threads=MAX_NUMBER_OF_THREADS)
print("Times per core for Spark for join query:")
for cores, t in times_per_core_join.items():
    print(f"Cores: {cores:2}, Time: {t:.4f} seconds")


In [ ]:
utils.plot_scalability(
    times_per_core_query_dict=times_per_core_agregation,
    title='Spark Aggregation Query Scalability'
)
utils.plot_scalability(
    times_per_core_query_dict=times_per_core_window,
    title='Spark Window Query Scalability'
)
utils.plot_scalability(
    times_per_core_query_dict=times_per_core_join,
    title='Spark Join Query Scalability'
)

### Task 2: Spark on Cluster

**Goal**: Compare Single Node performance vs. Spark on a Cluster.

**Instructions**:
1.  **Infrastructure**: Use the infrastructure from **Phase 1** (Google Dataproc). You may need to modify your Terraform code to adjust the cluster configuration (e.g., number of worker nodes).
2.  **Environment**: The easiest way to run this is via **Google Workbench** connected to your Dataproc cluster.
3.  **Upload Data**: Upload the generated `social_media_data.parquet` to HDFS or GCS.
    -   **Tip**: For better performance, consider **partitioning** the data (e.g., by `category` or `date`) when saving it to the distributed storage. This allows Spark to optimize reads.
4.  **Run Queries**: Run your PySpark queries from Task 1 on the cluster.
5.  **Scalability Test**: 
    -   Run the queries with different numbers of **worker nodes** (e.g., 2, 3, 4).
    -   You can achieve this by resizing the cluster (manually or via Terraform) or by configuring the number of executors in Spark.
6.  **Analyze**:
    -   How does the cluster performance compare to your local machine?
    -   Did adding more nodes/executors linearly improve performance?
    -   **Tip**: If Spark is slower than single-node engines, consider **increasing the dataset size** (e.g., generate 10M+ records or duplicate the data). Spark's overhead is significant for small data, and its true power appears when data exceeds single-node memory.

In [13]:
df = spark.read.parquet(TABLE_LOCATION)
df.write.partitionBy("category").mode("overwrite").parquet("social_media_data_partitioned.parquet")

We partitioned the file using the code above, then uploaded the partitioned data to the bucket using
`gsutil cp -r social_media_data_partitioned.parquet gs://tbd-2025z-318407-state/data/social_media_data_partitioned.parquet`.

Next, we created the file [dataproc_pyspark_test.py](https://github.com/mikic202/tbd-workshop-1-2025Z/blob/dev-tbd-workshop-pahse2/notebooks/dataproc_pyspark_test.py)
, which contains a slightly modified version of the functions used for local testing. Several adjustments were required to ensure the code would run correctly on Dataproc.

Finally, we submitted the job to our cluster using the following command:
`gcloud dataproc jobs submit pyspark dataproc_pyspark_test.py --cluster=tbd-cluster --region=europe-west1`.
Running this job produced the following results:

In [ ]:
mean_execution_times_aggr = {1: 10.4872, 2: 9.7069, 3: 7.4032, 4: 6.4003}
mean_execution_times_window = {1: 47.6567, 2: 38.1815, 3: 34.0026, 4: 25.2833}
mean_execution_times_join = {1: 65.2939, 2: 40.7196, 3: 29.7292, 4: 27.0644}

utils.plot_scalability(
    times_per_core_query_dict=mean_execution_times_aggr,
    title='PySpark on cluster Aggregation Query Scalability'
)

utils.plot_scalability(
    times_per_core_query_dict=mean_execution_times_window,
    title='PySpark on cluster Window Query Scalability'
)

utils.plot_scalability(
    times_per_core_query_dict=mean_execution_times_join,
    title='PySpark on cluster Join Query Scalability'
)

### Task 3: Execution Modes & Analysis

**Goal**: Deep dive into execution models and limitations.

**Instructions**:
1.  **Lazy vs. Eager vs. Streaming**:
    -   Use **Polars**. Compare the **Execution Time** and **Peak Memory** of:
        -   Eager execution (`read_parquet` -> filter).
        -   Lazy execution (`scan_parquet` -> filter -> `collect()`).
        -   Streaming execution (`scan_parquet` -> filter -> `collect(streaming=True)`).
2.  **Polars Limitations**:
    -   Identify a scenario where Polars might struggle compared to Spark (e.g., memory limits).
3.  **Decision Boundary**:
    -   Based on your findings, when would you recommend switching from a single-node tool (Polars/DuckDB) to a distributed engine (Spark)?

In [3]:
df_pl = pl.read_parquet(TABLE_LOCATION)

print(" --- Eager: --- ")
polars_tim_q1 = utils.benchmark_polars_time(df_pl, utils.query_aggregation_polars)
polars_mem_q1 = utils.benchmark_polars_memory(df_pl, utils.query_aggregation_polars)

df_pl = pl.scan_parquet(TABLE_LOCATION)

print(" --- Lazy: --- ")
polars_tim_q1 = utils.benchmark_polars_time(df_pl, utils.query_aggregation_polars_lazy)
polars_mem_q1 = utils.benchmark_polars_memory(df_pl, utils.query_aggregation_polars_lazy)


print(" --- Lazy Streaming: --- ")
polars_tim_q1 = utils.benchmark_polars_time(df_pl, utils.query_aggregation_polars_lazy_streaming)
polars_mem_q1 = utils.benchmark_polars_memory(df_pl, utils.query_aggregation_polars_lazy_streaming)


 --- Eager: --- 
Repeats number: 5
Execution times: [0.285, 0.206, 0.212, 0.234, 0.201]
First time: 0.285s
Mean  time: 0.227s
Best  time: 0.201s
Peak Memory Usage: 2366.801 MiB
 --- Lazy: --- 
Repeats number: 5
Execution times: [0.178, 0.175, 0.151, 0.191, 0.143]
First time: 0.178s
Mean  time: 0.168s
Best  time: 0.143s
Peak Memory Usage: 2498.668 MiB
Repeats number: 5
Execution times: [0.068, 0.05, 0.056, 0.059, 0.055]
First time: 0.068s
Mean  time: 0.058s
Best  time: 0.050s
Peak Memory Usage: 2613.812 MiB


2. **Polars limitations**

    - Polars, as a single-node tool, is limited to the machine's RAM capacity and free disk space, where it is run.
    So when the heavy dataset is loaded, the size that exceeds the machine's resources, Polar might not handle it and encounter *Out Of Memory Error*.

    - Current Polars version ([documentation <Jan 2026>](https://docs.pola.rs/user-guide/concepts/streaming/)) does not support streaming for every possible operation. When streaming is a key functionability in the given purpose, then using Polars might not be the best choice. 


3. **Decision Boundry** - switching from single-node tool to distributed one

    - Going for a distributed tool such as Spark is usefull especially when there are available distributed resources, such asaccess to clusters on cloud platforms.

    - If the dataset that is being processed is large and exceeds local machine's available resources then Spark have great power of running on many machines at once.

    - While the main purpose of using streaming in single-node tools is to save the memory, Spark on the other hand is prepared to process data in real time. If its a key feature then Spark is a go-to tool.

    - Polars is an emerging library and does not yet have the ecosystem and the same level of community and enterprise support as mature and established solutions such as Spark. 